In [ ]:
from pystac.client import Client
from odc.stac import load
import xarray as xr
from src.utils import mask_land, mask_deeps, make_indices, do_prediction, locations
import joblib
from tqdm import tqdm

from ipyleaflet import basemaps
import folium

from dep_tools.grids import PACIFIC_GRID_10

# import warnings
# warnings.filterwarnings("ignore")

In [ ]:
catalog = Client.open("https://earth-search.aws.element84.com/v1")
collection = "sentinel-2-l2a"

In [ ]:
# Set location using defined study locations
# location = locations.suva
# bbox = location.bbox

# Set location using tile
# tile_id = (130, 12)
# geobox = PACIFIC_GRID_10.tile_geobox(tile_id)
# bbox = list(geobox.geographic_extent.boundingbox)

# Set location using recent bath survey
ll = (-17.626335, 178.475445)
ur = (-17.562686, 178.563613)
bbox = [ll[1], ll[0], ur[1], ur[0]]

# Date
daterange = "2025-01/2025-03"

In [ ]:
items = catalog.search(
    collections=[collection],
    bbox=bbox,
    datetime=daterange,
    query={"eo:cloud_cover": {"lt": 50}},
).item_collection()

print(f"Found {len(items)} items")

In [ ]:
data = load(
    items,
    bbox=bbox,
    epsg="utm",
    measurements=[
        "scl",
        "nir",
        "red",
        "blue",
        "green",
        "nir08",
        "nir09",
        "swir16",
        "swir22",
        "coastal"
    ],
    chunks={"x": 2048, "y": 2048},
    nodata=0,
    groupby="solar_day"
)

# Mask clouds
mask = data.scl.isin([3, 8, 9, 10])
data = data.where(~mask)
data = make_indices(data)

# Mask land
data = mask_land(data)

# # Mask deep water
data = mask_deeps(data)

data = data.drop_vars("scl")

data

In [ ]:
from src.utils import load_model_bundle
from pathlib import Path

model, scaler = load_model_bundle(Path("models/2025_04_16d_nn_weights.zip"))

predictions_list = []

for day in tqdm(data.time):
    data_day = data.sel(time=day).compute()
    predictions_list.append(do_prediction(data_day, model, scaler=scaler))

# Concatenate them all together again
predictions = xr.concat(predictions_list, dim="time").to_dataset(name="elevation")

predictions

In [ ]:
from dea_tools.coastal import tidal_tag, pixel_tides

elev_with_tide = tidal_tag(predictions, model="FES2022", directory="/Users/alex/Data/tide_models_fiji/")
tides_highres, tides_lowres = pixel_tides(predictions, model="FES2022", directory="/Users/alex/Data/tide_models_fiji/", resample=True)
elev_with_tide["tide_m_pp"] = tides_highres
predictions["elv_tide_pp"] = predictions.elevation + elev_with_tide.tide_m_pp

In [ ]:
predictions["elv_tide_pp"] = predictions.elevation + elev_with_tide.tide_m_pp
predictions.elv_tide_pp.plot.imshow(col="time", col_wrap=2, cmap="Blues_r", robust=True, size=6)

In [ ]:
# Clean up the data by removing pixels that only had predictions sometimes
count = predictions.elevation.count(dim="time")
total = len(predictions.time)

mask = count > (total * 0.25)  # At least X% of the time there was a prediction

masked = predictions.where(mask)

mean = masked.elevation.mean(dim="time")
stdev = masked.elevation.std(dim="time")

mean_pp = masked.elv_tide_pp.mean(dim="time")
median_pp = masked.elv_tide_pp.median(dim="time")

# Make a fancy map
centroid = list(predictions.odc.geobox.geographic_extent.centroid.coords[0])[::-1]
m = folium.Map(location=centroid, zoom_start=12)
_ = folium.TileLayer(tiles=basemaps.Esri.WorldImagery).add_to(m)

count.odc.add_to(m, cmap="Reds", name="Count", enable=False)
stdev.odc.add_to(m, cmap="Reds", name="Stdev", enable=False)

mean.odc.add_to(m, cmap="Blues_r", name="Depth")
mean_pp.odc.add_to(m, cmap="Blues_r", name="Depth + Tide")
median_pp.odc.add_to(m, cmap="Blues_r", name="Depth + Tide (Median)")

folium.LayerControl().add_to(m)

m

In [ ]:
median_pp.odc.write_cog("median_pp.tif")

In [ ]:
mean_countmasked.odc.write_cog(f"{location.name}_deep_masked_30m.tif", overwrite=True)